# 6.7 · 谱聚类 / Spectral Clustering

> **课程定位 / Where this fits**
> K-Means(6.1)在非凸簇上失败。DBSCAN(6.4)靠密度。**谱聚类**走第三条路: 把数据建成**相似度图**, 用**图拉普拉斯矩阵的特征向量**把数据嵌入到一个新空间, 在那里非凸簇变得**线性可分**, 再跑 K-Means。它把"聚类"转化成"图分割", 数学优美, 能处理环形/月牙等任意形状。
> Spectral clustering builds a similarity graph, embeds points via the graph Laplacian's eigenvectors (where non-convex clusters become separable), then runs K-Means there.

> 💡 **面试相关 / Interview-relevant**
> - "谱聚类的流程 / 为什么用图拉普拉斯特征向量" ★★★★★
> - "图拉普拉斯 L=D-W 的性质 / 0特征值个数=连通分量数" ★★★★★
> - "谱聚类为什么能处理非凸簇" ★★★★
> - "谱聚类的缺点(O(n³)特征分解, 不可扩展)" ★★★★
> - "归一化拉普拉斯 vs 非归一化" ★★★

---

## 学习目标 / Learning Objectives
1. 相似度图 + 图拉普拉斯 $L=D-W$ 的性质。
2. 为何"取最小几个特征向量"能分图。
3. 从零实现谱聚类(亲和图→拉普拉斯→特征向量→K-Means)。
4. 在非凸数据上完胜 K-Means。
5. 缺点与可扩展性。

## 目录 / TOC
1. [图拉普拉斯与谱嵌入 ⭐](#1)
2. [⭕ 数据: 同心圆 + 从零实现 ⭐](#2)
3. [谱嵌入可视化 + vs K-Means ⭐](#3)
4. [缺点与超参](#4)
5. [小结](#5)


<a id="1"></a>
## 1. 图拉普拉斯与谱嵌入 ⭐ / Graph Laplacian & Spectral Embedding

**第一步, 建相似度图**: 点为节点, 边权 $W_{ij}$ = 相似度(常用高斯核 $\exp(-\|\mathbf{x}_i-\mathbf{x}_j\|^2/2\sigma^2)$ 或 k 近邻图)。近的点权大, 远的点权 ~0。

**第二步, 图拉普拉斯** $L = D - W$, 其中 $D$ 是度对角阵($D_{ii}=\sum_j W_{ij}$)。性质(面试核心):
- $L$ 对称半正定, 特征值 $0=\lambda_1\le\lambda_2\le\cdots$。
- **0 特征值的重数 = 图的连通分量数**, 对应的特征向量是各分量的指示向量。
- 若图近似有 K 个弱连通的块, 则**最小的 K 个特征向量**近似编码"哪个点属哪块"。

**第三步, 谱嵌入 + K-Means**: 取 $L$ 最小的 K 个特征向量作列, 每个点取这 K 维作新坐标; 在这个新空间里, 原本缠绕的非凸簇变成**分离的团**, 跑普通 K-Means 即可。

> 直觉: 谱聚类近似最小化**归一化割(Normalized Cut)**——把图切成内部连接强、彼此连接弱的块。


<a id="2"></a>
## 2. 数据: 同心圆 + 从零实现 ⭐ / Concentric Circles

**make_circles**: 两个同心环。环形是 K-Means/GMM 的彻底死穴(没有"球形质心"可言), 却是谱聚类的经典展示。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_circles
from scipy.linalg import eigh
sns.set_theme(style="whitegrid")

X, ytrue = make_circles(400, factor=0.4, noise=0.06, random_state=0)
print(f"同心圆: {X.shape}, 2 个环")

def spectral_scratch(X, K, sigma=0.2):
    # 1) 高斯核相似度图 W
    D2 = ((X[:,None,:]-X[None,:,:])**2).sum(-1)
    W = np.exp(-D2 / (2*sigma**2)); np.fill_diagonal(W, 0)
    # 2) 拉普拉斯 (对称归一化) L_sym = I - D^-1/2 W D^-1/2
    deg = W.sum(1); Dinv = np.diag(1/np.sqrt(deg))
    L = np.eye(len(X)) - Dinv @ W @ Dinv
    # 3) 取最小 K 个特征向量 (跳过的处理: 直接取前K)
    vals, vecs = eigh(L, subset_by_index=[0, K-1])
    U = vecs / (np.linalg.norm(vecs, axis=1, keepdims=True) + 1e-9)   # 行归一化
    # 4) 在谱嵌入上跑 K-Means
    from sklearn.cluster import KMeans
    return KMeans(K, n_init=10, random_state=0).fit_predict(U), U, vals

lab_scratch, U, vals = spectral_scratch(X, 2)
from sklearn.cluster import SpectralClustering
lab_sk = SpectralClustering(2, affinity="rbf", gamma=50, random_state=0).fit_predict(X)
from sklearn.metrics import adjusted_rand_score
print(f"从零谱聚类 与真实环 ARI: {adjusted_rand_score(ytrue, lab_scratch):.3f}")
print(f"sklearn 谱聚类 与真实环 ARI: {adjusted_rand_score(ytrue, lab_sk):.3f}")
print(f"拉普拉斯最小几个特征值: {vals[:4].round(4)} (接近0的个数≈连通块数)")


<a id="3"></a>
## 3. 谱嵌入可视化 + vs K-Means ⭐ / Embedding & vs K-Means


In [ ]:
from sklearn.cluster import KMeans
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
# 原始 + 谱聚类结果
axes[0].scatter(X[:,0], X[:,1], c=lab_scratch, cmap="coolwarm", s=15)
axes[0].set_title("谱聚类: 正确分出内外两环")
# 谱嵌入空间 (第2个特征向量值 vs 索引, 或两维)
axes[1].scatter(U[:,0], U[:,1], c=lab_scratch, cmap="coolwarm", s=15)
axes[1].set_title("谱嵌入空间: 两环变成两个分离的团(可线性分)")
axes[1].set_xlabel("特征向量1"); axes[1].set_ylabel("特征向量2")
# K-Means 直接做
axes[2].scatter(X[:,0], X[:,1], c=KMeans(2,n_init=10,random_state=0).fit_predict(X), cmap="coolwarm", s=15)
axes[2].set_title("K-Means 原空间: 球形切分 → 切错")
plt.tight_layout(); plt.show()
print("关键: 拉普拉斯特征向量把缠绕的环'展开'成分离团, 再普通 K-Means 即可")


<a id="4"></a>
## 4. 缺点与超参 / Limitations & Hyperparameters

- **缺点**: 需对 $n\times n$ 拉普拉斯做特征分解, **$O(n^3)$**, 大数据不可行(可用 Nyström/稀疏近似)。还要预设簇数 K。
- **超参**: 相似度图的尺度最关键——RBF 的 `gamma`/`sigma` 或 kNN 图的 k。太大/太小都会让图结构失真。下面看 gamma 的敏感性。


In [ ]:
from sklearn.cluster import SpectralClustering
from sklearn.metrics import adjusted_rand_score
print("谱聚类对相似度尺度(gamma)敏感:")
for g in [1, 5, 12, 50, 200]:
    lab = SpectralClustering(2, affinity="rbf", gamma=g, random_state=0).fit_predict(X)
    print(f"  gamma={g:>3}: 与真实环 ARI = {adjusted_rand_score(ytrue, lab):.3f}")
print("gamma 太小→图太密(全连一块); 太大→图太疏(断成碎块); 需调到合适尺度")


<a id="5"></a>
## 5. 小结 / Summary

```
谱聚类: 相似度图 W → 拉普拉斯 L=D-W → 取最小 K 个特征向量做嵌入 → 在嵌入上 K-Means
图拉普拉斯性质: 半正定; 0特征值重数=连通分量数; 最小K个特征向量编码块归属
非凸簇(环/月牙)在谱嵌入里变成分离团 → 线性可分 → 普通 K-Means 搞定
近似最小化归一化割(NCut); 对相似度尺度(gamma/k)敏感
缺点: O(n³) 特征分解不可扩展; 需预设 K
```

### 💡 面试速查
1. **流程**: 建相似度图→图拉普拉斯→最小K个特征向量嵌入→K-Means
2. **L=D-W 半正定**; **0特征值个数=连通分量数**(谱聚类的理论基石)
3. **能处理非凸簇**: 特征向量把缠绕结构展开成线性可分
4. **缺点 O(n³)** 不可扩展; 对相似度图尺度敏感; 需预设 K
5. 本质是**图分割/归一化割**问题

### 下一节
**6.8 PCA**——从聚类转到降维。PCA 是最经典的线性降维: 找方差最大的正交方向, 用最少维度保留最多信息, 是无监督学习另一大支柱。
